# 03 · Context Managers & the `with` Statement

**Module 09 – Advanced Functions** · Estimated time: 40 min

> Expands 📗 *BONUS – With Statement Context Managers* (which only shows files) with writing your own context managers, `contextlib`, and async/multiple contexts.

### 🎯 What you will learn
- The problem: resources that must be released (files, locks, connections)
- `try/finally` vs `with` (📗)
- How `with` works: `__enter__` / `__exit__`
- Writing class-based context managers
- `@contextlib.contextmanager`: generator-based context managers
- Useful `contextlib` tools: `suppress`, `redirect_stdout`, `ExitStack`, `chdir`
- Multiple managers in one `with` (parenthesised, 3.10+)

## 1. The problem (📗)

In [1]:
p = open("oops.txt", "a")
try:
    p.readlines()             # opened for appending, so reading fails
except Exception as e:
    print(type(e).__name__, e)
print("Is the file closed?", p.closed)   # ❌ still open: a leaked resource
p.close()

UnsupportedOperation not readable
Is the file closed? False


### Protect the file with `try/except/finally` (📗)

In [2]:
p = open("oops.txt", "a")
try:
    p.readlines()
except Exception:
    print("An exception was raised!")
finally:
    p.close()
print("Closed?", p.closed)

An exception was raised!
Closed? True


### ✅ Save steps with `with`

In [3]:
try:
    with open("oops.txt", "a") as p:
        p.readlines()
except Exception as e:
    print("Error:", type(e).__name__)
print("Closed?", p.closed)          # closed automatically, even though an error happened

Error: UnsupportedOperation
Closed? True


## 2. How `with` works

```python
with EXPR as VAR:
    BLOCK
```
roughly means:
```python
manager = EXPR
VAR = manager.__enter__()
try:
    BLOCK
except BaseException as e:
    if not manager.__exit__(type(e), e, e.__traceback__):
        raise                       # re-raise unless __exit__ returned True
else:
    manager.__exit__(None, None, None)
```

- `__enter__` sets things up and returns the object bound by `as`
- `__exit__` **always** runs and cleans up; returning `True` would *suppress* the exception (rarely what you want)

## 3. Class-based context managers

In [4]:
import time

class Timer:
    def __init__(self, label="block"):
        self.label = label

    def __enter__(self):
        self.start = time.perf_counter()
        return self                          # available as `as t`

    def __exit__(self, exc_type, exc_value, traceback):
        self.elapsed = time.perf_counter() - self.start
        status = "failed" if exc_type else "finished"
        print(f"[{self.label}] {status} in {self.elapsed:.4f}s")
        return False                         # don't swallow exceptions

with Timer("sum") as t:
    total = sum(range(2_000_000))
print(total, round(t.elapsed, 4))

try:
    with Timer("crash"):
        1 / 0
except ZeroDivisionError:
    print("exception still propagated")

[sum] finished in 0.0166s
1999999000000 0.0166
[crash] failed in 0.0000s
exception still propagated


### Example: a "transaction" that rolls back on error

In [5]:
class Transaction:
    def __init__(self, account):
        self.account = account
    def __enter__(self):
        self.snapshot = dict(self.account)       # save state
        return self.account
    def __exit__(self, exc_type, exc, tb):
        if exc_type is not None:
            self.account.clear()
            self.account.update(self.snapshot)   # roll back
            print("Rolled back:", exc)
            return True                          # swallow the error on purpose here
        print("Committed")

account = {"balance": 100}
with Transaction(account) as acc:
    acc["balance"] -= 30
with Transaction(account) as acc:
    acc["balance"] -= 500
    if acc["balance"] < 0:
        raise ValueError("insufficient funds")
print(account)

Committed
Rolled back: insufficient funds
{'balance': 70}


## 4. ✅ `@contextlib.contextmanager`: the easy way

Write a **generator** that yields once: code before `yield` = `__enter__`, code after = `__exit__`. Wrap the `yield` in `try/finally` so cleanup always happens.

In [6]:
from contextlib import contextmanager

@contextmanager
def timer(label):
    start = time.perf_counter()
    try:
        yield                                  # the with-block runs here
    finally:
        print(f"[{label}] {time.perf_counter() - start:.4f}s")

with timer("list build"):
    data = [i ** 2 for i in range(500_000)]

[list build] 0.0133s


In [7]:
import os

@contextmanager
def temp_file(name, content):
    with open(name, "w", encoding="utf-8") as f:
        f.write(content)
    try:
        yield name                             # value for `as`
    finally:
        os.remove(name)
        print(f"(deleted {name})")

with temp_file("scratch.txt", "temporary data") as path:
    print(open(path, encoding="utf-8").read())
print(os.path.exists("scratch.txt"))

temporary data
(deleted scratch.txt)
False


## 5. Handy tools in `contextlib`

In [8]:
from contextlib import suppress, redirect_stdout, ExitStack, chdir
import io

# suppress: ignore specific exceptions
with suppress(FileNotFoundError):
    os.remove("does_not_exist.txt")
print("no crash")

# redirect_stdout: capture printed output
buffer = io.StringIO()
with redirect_stdout(buffer):
    print("captured!")
print("buffer contains:", repr(buffer.getvalue()))

# chdir (3.11+): temporarily change the working directory
os.makedirs("subdir", exist_ok=True)
with chdir("subdir"):
    print("inside:", os.path.basename(os.getcwd()))
print("back in:", os.path.basename(os.getcwd()))

no crash
buffer contains: 'captured!\n'
inside: subdir
back in: tmp6kll950n


In [9]:
# ExitStack: a dynamic number of context managers
names = ["a.txt", "b.txt", "c.txt"]
for n in names:
    with open(n, "w", encoding="utf-8") as f:
        f.write(n.upper())
with ExitStack() as stack:
    files = [stack.enter_context(open(n, encoding="utf-8")) for n in names]
    print([f.read() for f in files])
print(all(f.closed for f in files))

['A.TXT', 'B.TXT', 'C.TXT']
True


## 6. Multiple context managers in one `with`

In [10]:
with open("a.txt", encoding="utf-8") as src, open("copy.txt", "w", encoding="utf-8") as dst:
    dst.write(src.read())

# 3.10+: parentheses allow one manager per line
with (
    open("a.txt", encoding="utf-8") as first,
    open("b.txt", encoding="utf-8") as second,
):
    print(first.read(), second.read())

A.TXT B.TXT


## 7. Context managers you'll meet everywhere

| Code | Manages |
|---|---|
| `with open(...)` | files |
| `with threading.Lock():` | locks (Module 13) |
| `with sqlite3.connect(...) as conn:` | DB transactions (commit/rollback) |
| `with requests.Session() as s:` | HTTP connection pools |
| `with zipfile.ZipFile(...)` | archives |
| `with tempfile.TemporaryDirectory() as d:` | temporary folders |
| `with decimal.localcontext() as ctx:` | precision settings |
| `with warnings.catch_warnings():` | warning filters |
| `async with ...` | async resources (Module 13) |

## 📝 Summary
- `with` guarantees cleanup, replacing `try/finally` boilerplate.
- Class-based: implement `__enter__` and `__exit__(exc_type, exc, tb)`.
- Generator-based: `@contextmanager` + `try: yield finally:`.
- `contextlib` gives `suppress`, `redirect_stdout`, `chdir`, `ExitStack`.

## 🏋️ Practice
1. Write a context manager `cd(path)` with `@contextmanager` (without using `contextlib.chdir`).
2. Write a class `Indenter` so nested `with` blocks print with increasing indentation:
   ```python
   with Indenter() as ind:
       ind.print("hi")
       with ind:
           ind.print("hello")
   ```
3. Write `@contextmanager def open_upper(path)` that yields the file's content in uppercase.

<details><summary>Show solutions</summary>

```python
import os
from contextlib import contextmanager

@contextmanager
def cd(path):
    old = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old)

class Indenter:
    def __init__(self): self.level = -1
    def __enter__(self):
        self.level += 1
        return self
    def __exit__(self, *exc): self.level -= 1
    def print(self, text): print("    " * self.level + text)

@contextmanager
def open_upper(path):
    with open(path, encoding="utf-8") as f:
        yield f.read().upper()
```
</details>